# 03 · Feature Engineering
**Project:** EV Charging Demand & Cost Analytics + Predictive System  
**Input:** `data/processed/ev_charging_cleaned.csv`  
**Output:** `data/processed/ev_charging_features.csv`

---

### Engineering Roadmap
| Feature | Logic |
|------|--------|
| `soc_gained_pct` | State of Charge (End %) - State of Charge (Start %) |
| `charging_efficiency` | Energy Consumed (kWh) / Charging Duration (hours) |
| `cost_per_kwh` | Charging Cost (USD) / Energy Consumed (kWh) |
| `battery_utilization_ratio` | Energy Consumed (kWh) / Battery Capacity (kWh) |
| Time Features | `hour_of_day`, `month`, `is_weekend` (extracted from Charging Start Time / Day of Week) |
| `distance_per_kwh` | Distance Driven (km) / Energy Consumed (kWh) (with div-by-zero handling) |
| `charger_type_ordinal` | Ordinal encoding of Charger Type (Level 1 = 1, Level 2 = 2, DC Fast = 3) |
| Export | Save to `/data/processed/ev_charging_features.csv` |

---

## 0 · Imports & Load Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

CLEANED_PATH = '../data/processed/ev_charging_cleaned.csv'
FEATURES_PATH = '../data/processed/ev_charging_features.csv'

df = pd.read_csv(CLEANED_PATH)
print(f'Loaded cleaned data: {df.shape[0]:,} rows × {df.shape[1]} columns')

cols_before = df.columns.tolist()
print('\nColumns BEFORE feature engineering:')
for i, col in enumerate(cols_before, 1):
    print(f'  {i:2d}. {col}')

Loaded cleaned data: 1,320 rows × 26 columns

Columns BEFORE feature engineering:
   1. User ID
   2. Vehicle Model
   3. Battery Capacity (kWh)
   4. Charging Station ID
   5. Charging Station Location
   6. Charging Start Time
   7. Charging End Time
   8. Energy Consumed (kWh)
   9. Charging Duration (hours)
  10. Charging Rate (kW)
  11. Charging Cost (USD)
  12. Time of Day
  13. Day of Week
  14. State of Charge (Start %)
  15. State of Charge (End %)
  16. Distance Driven (since last charge) (km)
  17. Temperature (C)
  18. Vehicle Age (years)
  19. Charger Type
  20. User Type
  21. soc_was_swapped
  22. start_hour
  23. start_month
  24. start_dow_num
  25. is_weekend
  26. soc_delta_pp


---
## 1 · Engineer New Features

In [2]:
df_feat = df.copy()

# 1. soc_gained_pct
df_feat['soc_gained_pct'] = df_feat['State of Charge (End %)'] - df_feat['State of Charge (Start %)']

# 2. charging_efficiency (Effective kW rate)
df_feat['charging_efficiency'] = df_feat['Energy Consumed (kWh)'] / df_feat['Charging Duration (hours)']

# 3. cost_per_kwh
# Prevent div-by-zero just in case (though we validated Energy > 0 in cleaning)
df_feat['cost_per_kwh'] = np.where(df_feat['Energy Consumed (kWh)'] > 0, 
                                   df_feat['Charging Cost (USD)'] / df_feat['Energy Consumed (kWh)'], 
                                   np.nan)

# 4. battery_utilization_ratio
df_feat['battery_utilization_ratio'] = np.where(df_feat['Battery Capacity (kWh)'] > 0,
                                                df_feat['Energy Consumed (kWh)'] / df_feat['Battery Capacity (kWh)'],
                                                np.nan)

# 5. Time Features: hour_of_day, month, is_weekend
df_feat['Charging Start Time'] = pd.to_datetime(df_feat['Charging Start Time'])
df_feat['hour_of_day'] = df_feat['Charging Start Time'].dt.hour
df_feat['month'] = df_feat['Charging Start Time'].dt.month

# For is_weekend, we can use 'Day of Week' column or the datetime object
df_feat['is_weekend'] = df_feat['Day of Week'].isin(['Saturday', 'Sunday']).astype(int)

# 6. distance_per_kwh
df_feat['distance_per_kwh'] = np.where(df_feat['Energy Consumed (kWh)'] > 0,
                                       df_feat['Distance Driven (since last charge) (km)'] / df_feat['Energy Consumed (kWh)'],
                                       np.nan)

# Replace any infinite values that might have crept in with 0
for col in ['cost_per_kwh', 'battery_utilization_ratio', 'distance_per_kwh', 'charging_efficiency']:
    df_feat[col].replace([np.inf, -np.inf], np.nan, inplace=True)
    df_feat[col].fillna(0, inplace=True)

# 7. Ordinal encoding for Charger Type
charger_map_lower = {'level 1': 1, 'level 2': 2, 'dc fast charger': 3}
df_feat['charger_type_ordinal'] = df_feat['Charger Type'].str.lower().map(charger_map_lower)

print('New features engineered successfully.')

New features engineered successfully.


C:\Users\sskor\AppData\Local\Temp\ipykernel_17332\2029503325.py:35: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_feat[col].replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\sskor\AppData\Local\Temp\ipykernel_17332\2029503325.py:36: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through cha

---
## 2 · Validation & Summary

In [3]:
cols_after = df_feat.columns.tolist()
print(f'Columns AFTER feature engineering: {len(cols_after)}')
new_cols = [c for c in cols_after if c not in cols_before]
print(f'\nNewly added columns ({len(new_cols)}):')
for i, col in enumerate(new_cols, 1):
    print(f'  {i}. {col}')

print('\n\n--- Summary Statistics of New Numeric Features ---')
numeric_new_cols = ['soc_gained_pct', 'charging_efficiency', 'cost_per_kwh', 
                    'battery_utilization_ratio', 'hour_of_day', 'month', 
                    'is_weekend', 'distance_per_kwh', 'charger_type_ordinal']

display(df_feat[numeric_new_cols].describe().T.rename(columns={'50%': 'median'}))

Columns AFTER feature engineering: 34

Newly added columns (8):
  1. soc_gained_pct
  2. charging_efficiency
  3. cost_per_kwh
  4. battery_utilization_ratio
  5. hour_of_day
  6. month
  7. distance_per_kwh
  8. charger_type_ordinal


--- Summary Statistics of New Numeric Features ---


,count,mean,std,min,25%,median,75%,max
soc_gained_pct,1320.0000,31.6379,22.1947,0.0008,12.8555,27.0445,47.7218,92.8257
charging_efficiency,1320.0000,26.4680,34.4508,0.0224,10.9277,18.9507,30.2658,784.1352
cost_per_kwh,1320.0000,1.6046,19.9513,0.0045,0.3002,0.5378,0.9188,654.7621
battery_utilization_ratio,1320.0000,0.6843,1.0878,0.0005,0.3407,0.5602,0.8382,27.7049
hour_of_day,1320.0000,11.5000,6.9248,0.0000,5.7500,11.5000,17.2500,23.0000
month,1320.0000,1.4364,0.4961,1.0000,1.0000,1.0000,2.0000,2.0000
is_weekend,1320.0000,0.3000,0.4584,0.0000,0.0000,0.0000,1.0000,1.0000
distance_per_kwh,1320.0000,10.9287,164.9090,0.0187,1.8886,3.5415,6.7336,5962.5690
charger_type_ordinal,1320.0000,1.9780,0.8207,1.0000,1.0000,2.0000,3.0000,3.0000


In [4]:
# Drop redundant columns from data cleaning phase to keep things clean
# 'start_hour', 'start_month', 'start_dow_num', 'soc_delta_pp' were derived in step 2 but we just 
# recreated them with the names requested by the user, so we can drop the old ones.
redundant = ['start_hour', 'start_month', 'start_dow_num', 'soc_delta_pp']
cols_to_drop = [c for c in redundant if c in df_feat.columns]
if cols_to_drop:
    df_feat.drop(columns=cols_to_drop, inplace=True)
    print(f'Dropped redundant old columns: {cols_to_drop}')
    
print(f'Final shape: {df_feat.shape[0]:,} rows × {df_feat.shape[1]} columns')

Dropped redundant old columns: ['start_hour', 'start_month', 'start_dow_num', 'soc_delta_pp']
Final shape: 1,320 rows × 30 columns


---
## 3 · Save Enriched Dataset

In [5]:
import os
os.makedirs('../data/processed', exist_ok=True)

df_feat.to_csv(FEATURES_PATH, index=False)

file_size_kb = os.path.getsize(FEATURES_PATH) / 1024
print(f' Enriched dataset saved to: {FEATURES_PATH}')
print(f'   Rows    : {len(df_feat):,}')
print(f'   Columns : {df_feat.shape[1]}')
print(f'   File    : {file_size_kb:.1f} KB')

 Enriched dataset saved to: ../data/processed/ev_charging_features.csv
   Rows    : 1,320
   Columns : 30
   File    : 499.5 KB
